# XAI und Fairness · Handrechnungen prüfen

Begleitnotebook zu [[shapley-von-hand]], [[fairness-metriken-rechnen]], [[logistische-regression-odds-ratio]] und [[bootstrap-oob-anteil]].

In [1]:
import itertools, math
import numpy as np

## 1 · Exakte Shapley-Werte (Formel aus VL6, Folie 7)

In [2]:
def shapley(v, p):
    w = lambda k: math.factorial(k) * math.factorial(p - k - 1) / math.factorial(p)
    phi = {}
    for i in range(1, p + 1):
        others = [j for j in range(1, p + 1) if j != i]
        phi[i] = sum(w(len(S)) * (v[tuple(sorted(S + (i,)))] - v[S])
                     for k in range(p) for S in itertools.combinations(others, k))
    return phi

v = {(): 10, (1,): 14, (2,): 12, (3,): 10, (1, 2): 18, (1, 3): 15, (2, 3): 13, (1, 2, 3): 20}
phi = shapley(v, 3)
print("Shapley-Werte:", phi)
print("Effizienz: Summe =", sum(phi.values()), "| f(N) - f(leer) =", v[(1, 2, 3)] - v[()])

Shapley-Werte: {1: 5.5, 2: 3.5, 3: 1.0}
Effizienz: Summe = 10.0 | f(N) - f(leer) = 10


### Lineares Modell mit interventionaler Baseline
Für $f(x) = \beta_0 + \beta^Tx$ und Baseline $\mathbb E[x]$ gilt $\phi_j = \beta_j(x_j - \mathbb E[x_j])$.

In [3]:
beta0, beta = 1.0, np.array([2.0, -1.0])
x, mu = np.array([3.0, 1.0]), np.array([1.0, 2.0])

def f_S(S):
    z = mu.copy()
    for j in S:
        z[j - 1] = x[j - 1]
    return beta0 + beta @ z

v_lin = {S: f_S(S) for k in range(3) for S in itertools.combinations([1, 2], k)}
print("Shapley:", shapley(v_lin, 2), "| Formel:", beta * (x - mu))

Shapley: {1: np.float64(4.0), 2: np.float64(1.0)} | Formel: [4. 1.]


## 2 · Fairness-Metriken je Gruppe

In [4]:
groups = {"A": dict(TP=40, FN=10, FP=20, TN=130), "B": dict(TP=15, FN=15, FP=5, TN=65)}
for g, c in groups.items():
    n = sum(c.values()); p = (c["TP"] + c["FN"]) / n
    tpr = c["TP"] / (c["TP"] + c["FN"]); fpr = c["FP"] / (c["FP"] + c["TN"]); ppv = c["TP"] / (c["TP"] + c["FP"])
    sel = (c["TP"] + c["FP"]) / n
    ident = p / (1 - p) * (1 - ppv) / ppv * tpr
    print(f"{g}: Basisrate {p:.3f} | Auswahlrate {sel:.3f} | TPR {tpr:.3f} | FPR {fpr:.3f} | PPV {ppv:.3f} | Identität FPR {ident:.3f}")

A: Basisrate 0.250 | Auswahlrate 0.300 | TPR 0.800 | FPR 0.133 | PPV 0.667 | Identität FPR 0.133
B: Basisrate 0.300 | Auswahlrate 0.200 | TPR 0.500 | FPR 0.071 | PPV 0.750 | Identität FPR 0.071


## 3 · Odds Ratio: gleicher Koeffizient, verschiedene Wahrscheinlichkeitseffekte

In [5]:
OR = math.exp(0.7)
print("Odds Ratio:", round(OR, 3))
for p0 in [0.1, 0.5, 0.9]:
    o = p0 / (1 - p0) * OR
    print(f"p={p0}: neu {o/(1+o):.3f}, Änderung {o/(1+o) - p0:+.3f}")

Odds Ratio: 2.014
p=0.1: neu 0.183, Änderung +0.083
p=0.5: neu 0.668, Änderung +0.168
p=0.9: neu 0.948, Änderung +0.048


## 4 · Bootstrap: Anteil Out-of-Bag (Simulation vs. Formel)

In [6]:
rng = np.random.default_rng(0)
for n in [5, 10, 100, 1000]:
    sim = np.mean([1 - len(np.unique(rng.integers(0, n, n))) / n for _ in range(2000)])
    print(f"n={n:>4}: simuliert {sim:.3f} | Formel (1-1/n)^n = {(1 - 1/n)**n:.3f}")
print("Grenzwert e^-1 =", round(math.exp(-1), 3))

n=   5: simuliert 0.325 | Formel (1-1/n)^n = 0.328
n=  10: simuliert 0.344 | Formel (1-1/n)^n = 0.349


n= 100: simuliert 0.365 | Formel (1-1/n)^n = 0.366


n=1000: simuliert 0.368 | Formel (1-1/n)^n = 0.368
Grenzwert e^-1 = 0.368
